# TensorFly — Run all

1. Select a paid GPU runtime (A100 preferred). 2. Click **Runtime → Run all**.

The notebook automatically clones/updates TensorFly, installs it in this kernel, downloads and verifies official MaleCNS v1.0 data, runs Qwen3.5-9B experiments and equal-budget baselines, then opens the real-morphology replay viewer. No manual file upload or environment variable is required.


In [ ]:
# SETUP — run automatically by Runtime → Run all
from pathlib import Path
import os, subprocess, sys

repo = Path('/content/fly-inference-optimizer')
if (repo / '.git').is_dir():
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only', 'origin', 'main'], check=True)
elif repo.exists():
    raise RuntimeError(f'{repo} exists but is not a TensorFly git clone. Use a clean Colab runtime.')
else:
    subprocess.run(['git', 'clone', '--branch', 'main', 'https://github.com/MrFaruk0/fly-inference-optimizer.git', str(repo)], check=True)
os.chdir(repo)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', '--no-deps'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'pyarrow', 'accelerate', 'safetensors', 'transformers'], check=True)
# Do not depend on Colab refreshing editable-install metadata: import this checkout directly.
source_root = str(repo / 'src')
if source_root not in sys.path:
    sys.path.insert(0, source_root)

import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select a paid CUDA GPU runtime, then use Runtime → Run all again.')
import tensorfly
print('TensorFly:', tensorfly.__version__)
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# COMPLETE EXPERIMENT — downloads real MaleCNS, runs Qwen, compares methods, exports replay
import subprocess
from tensorfly import DEFAULT_PROMPTS, TensorFlyExperiment

prepared = tensorfly.prepare()
experiment = TensorFlyExperiment(model='Qwen/Qwen3.5-9B')
tensorfly_records = experiment.run(prompt_corpus=DEFAULT_PROMPTS, trials=20)
baselines = experiment.compare_baselines(prompt_corpus=DEFAULT_PROMPTS, trials=20)
replay_path = experiment.export_video()

summary = {
    name: {
        'final_reward': rows[-1].reward,
        'mean_reward': sum(row.reward for row in rows) / len(rows),
        'final_config': rows[-1].resulting_next_config,
    }
    for name, rows in baselines.items()
}
print('MaleCNS report:', prepared.dataset.report)
print('Baseline summary:', summary)
print('Replay:', replay_path)

# Open the actual-SWC viewer in Colab; it consumes only recorded replay states.
server = subprocess.Popen([sys.executable, '-m', 'http.server', '8000', '--directory', 'viewer'])
from google.colab import output
output.serve_kernel_port_as_window(8000)
